# 3장 : 텐서 구조체

 + 파이토치 기본 자료구조인 텐서의 이해
 + 텐서를 인덱스로 접근해서 연산하기
 + 다차원 넘파이 배열과 연계해서 다루기
 + 성능 개선을 위해 GPU로 연산 처리

3장에서는 파이토치의 텐서를 통해, 입력받은 데이터를 부동소수점 수로 변환(floating point number)하는 방법에 대해 다룬다.

## 3.1 부동소수점 수의 세계

  - 신경망 사용의 핵심 : 실세계의 데이터를 신경망이 연산 가능하도록 인코딩해야 하고, 처리 후 나온 결과를 이해할 수 있게 혹은 사용할 수 있도록 디코딩해야 함
  - 위의 인코딩과 디코딩 사이의 중간과정에서 신경망이 입력에 대해 출력으로 표현되는 방법을 설명하는 중간 표현값들은 모두 부동소수점 수로 이루어짐
  - 이러한 중간 표현값은 사실 입력과 이전 층의 뉴런이 가진 가중치를 조합한 결과

  - $\textbf{텐서}^{\text{tensor}}$ : 데이터 처리와 저장을 위해 사용하는 파이토치의 기본 자료구조
    + 딥러닝에서의 텐서는 임의의 차원을 가진 벡터나 행렬의 일반화된 개념으로 생각하면 됨
    + $\textbf{다차원 배열}^{\text{multidimensional array}}$ 이라고도 부름.
    + 텐서의 차원 개수 = 텐서 안의 스칼라 값 참조를 위해 사용하는 인덱스 개수

  - $\textbf{넘파이}^{\text{Numpy}}$ : 다차원 배열을 다루는 데이터과학 분야의 표준 라이브러리.
    + 파이토치는 넘파이와 깔끔하게 호환되도록 만들어져있음(`reshape`, `dtype`, `squeeze`등 동일하거나 유사한 기능을 하는 메서드가 많음)
    + $\text{사이파이}^{\text{SciPy}}$, $\text{사이킷런}^{\text{Scikit-learn}}$, $\text{판다스}^{\text{Pandas}}$ 같은 과학 라이브러리와 유연한 통합 가능

## 3.2 텐서 : 다차원 배열

### 3.2.1 파이썬 리스트에서 파이토치 텐서로

In [1]:
# `list` 인덱싱 예제
a = [1.0, 2.0, 1.0]
print(a[0])
print(a[2])
print(a)

1.0
1.0
[1.0, 2.0, 1.0]


### 3.2.2 첫 텐서 만들어보기

$\textbf{NOTE}$

17:10:17.934 [error] Disposing session as kernel process died ExitCode: 3, Reason: OMP: Error #15: Initializing libiomp5md.dll, but found libiomp5md.dll already initialized.
OMP: Hint This means that multiple copies of the OpenMP runtime have been linked into the program. That is dangerous, since it can degrade performance or cause incorrect results. The best thing to do is to ensure that only a single OpenMP runtime is linked into the process, e.g. by avoiding static linking of the OpenMP runtime in any library. As an unsafe, unsupported, undocumented workaround you can set the environment variable KMP_DUPLICATE_LIB_OK=TRUE to allow the program to continue to execute, but that may cause crashes or silently produce incorrect results. For more information, please see http://www.intel.com/software/products/support/.

> 위와 같은 오류는 openmp와 torch가 충돌을 일으키면서 생기는 문제 

> -> 우선 pip uninstall -y intel-openmp 로 openmp 지운다음, conda forge로 재설치

> conda install intel-openmp

In [2]:
import torch

a = torch.ones(3)
a

tensor([1., 1., 1.])

In [3]:
print(a[1])
print(float(a[1]))
a[2] = 2.0
print(a)

tensor(1.)
1.0
tensor([1., 1., 2.])


### 3.2.3 텐서의 핵심

$\textbf{다차원 배열 vs 리스트, 튜플}$

  - 숫자값으로 만든 파이썬 리스트나 튜플 객체는 메모리에 따로따로 할당(포인터)
  - 파이토치 텐서나 넘파이 배열은 파이썬 객체가 아닌 $\textbf{언박싱}^{\text{unboxing}}$된 C 언어의 숫자 타입을 포함한 연속적인 메모리가 할당되고 이에 대한 뷰 제공
  - 각 요소는 (default 기준) 32비트(4바이트) `float`타입이며, 100만 개의 `float`타입숫자를 1차원 텐서에 보관하는 경우 400만 바이트의 연속적인 공간과 메타데이터 공간을 조금 더 차지하게 됨

In [4]:
# 좌표 (4,1), (5,3), (2,1)을 텐서로 저장하는 경우, 
# 파이썬 리스트로 숫자 좌표를 유지하는 대신 짝수 인덱스는 X좌표에 해당하게, 홀수 인덱스는 Y좌표에 해당하게 넣을 수 있다
points = torch.zeros(6)
x_y_list = [4.0, 1.0, 5.0, 3.0, 2.0, 1.0]
for n in range(len(x_y_list)):
    points[n] = x_y_list[n]
points

tensor([4., 1., 5., 3., 2., 1.])

In [5]:
# 혹은 리스트 자체를 텐서에 담아도 된다(생성자에 파이썬 리스트 넘기기)
points = torch.tensor([4.0, 1.0, 5.0, 3.0, 2.0, 1.0])
points

tensor([4., 1., 5., 3., 2., 1.])

In [6]:
# 첫 번째 점의 좌표 읽기
float(points[0]), float(points[1])

(4.0, 1.0)

In [7]:
# 위와 같은 방식은 번거로우므로 2차원 좌표를 바로 인덱싱할 수 있는 2차원 텐서를 사용한다
points = torch.tensor([[4.0, 1.0], [5.0, 3.0], [2.0, 1.0]])
points

tensor([[4., 1.],
        [5., 3.],
        [2., 1.]])

In [8]:
# 혹은 reshape도 사용가능하다.
points = torch.tensor([4.0, 1.0, 5.0, 3.0, 2.0, 1.0]).reshape(3,2)
points

tensor([[4., 1.],
        [5., 3.],
        [2., 1.]])

In [9]:
# reshape 예제 (2)
points = torch.tensor([4.0, 1.0, 5.0, 3.0, 2.0, 1.0]).reshape(3,-1)
points

tensor([[4., 1.],
        [5., 3.],
        [2., 1.]])

In [10]:
# 텐서의 차원 살펴보기 : shape 메서드를 통해 각 차원과 함께 텐서의 크기도 알 수 있다.
points.shape

torch.Size([3, 2])

In [11]:
# 텐서 초기화를 위해 차원별 크기 정보를 튜플로 만들어 zeros나 ones로 넘겨주기
points = torch.zeros(3,2)
points

tensor([[0., 0.],
        [0., 0.],
        [0., 0.]])

In [12]:
# 두 개의 인덱스를 사용한 2차원 텐서 내 요소의 개별적 접근
points = torch.tensor([[4.0, 1.0], [5.0, 3.0], [2.0, 1.0]])
points

tensor([[4., 1.],
        [5., 3.],
        [2., 1.]])

In [13]:
points[0,1]

tensor(1.)

In [14]:
points[0]

tensor([4., 1.])

## 3.3 텐서 인덱싱

리스트와 텐서의 인덱싱 차이

In [15]:
some_list = list(range(6))
print(some_list[:])
print(some_list[1:4])
print(some_list[:4])
print(some_list[:-1])
print(some_list[1:4:2])

[0, 1, 2, 3, 4, 5]
[1, 2, 3]
[0, 1, 2, 3]
[0, 1, 2, 3, 4]
[1, 3]


In [16]:
print(points[1:])
print(points[1:, :])
print(points[1:, 0])
print(points[None]) # 길이가 1인 차원 추가. unsqueeze와 동일 (Numpy방식과 매우 유사)

tensor([[5., 3.],
        [2., 1.]])
tensor([[5., 3.],
        [2., 1.]])
tensor([5., 2.])
tensor([[[4., 1.],
         [5., 3.],
         [2., 1.]]])


## 3.4 이름이 있는 텐서

텐서는 차원이나 축이 있고, 각 차원은 픽셀 위치나 컬러 채널에 해당한다.

따라서 텐서에 접근하려면 차원의 순서를 기억해서 인덱싱해야 한다.

In [17]:
# 3차원 텐서의 예 : 이미지 데이터
img_t = torch.randn(3, 5, 5) # 각각이 [채널 크기, 행 크기, 열 크기]가 된다
weights = torch.tensor([0.2126, 0.7152, 0.0722])

# 배치의 예 : 개별 이미지 여러 개를 하나의 텐서에 묶기
batch_t = torch.randn(2, 3, 5, 5) # [배치 크기, 채널 크기, 행 크기, 열 크기]

In [18]:
# RGB 채널은 주로 0번 혹은 1번 차원에 있고, 
# 두 경우 모두 뒤에서부터 세 번째 차원이므로 RGB 채널은 -3번 차원에 있는 것으로 일반화 가능
img_gray_naive = img_t.mean(-3)
batch_gray_naive = batch_t.mean(-3)
img_gray_naive.shape, batch_gray_naive.shape

(torch.Size([5, 5]), torch.Size([2, 5, 5]))

In [19]:
unsqueezed_weights = weights.unsqueeze(-1).unsqueeze(-1)
print(unsqueezed_weights.shape)

torch.Size([3, 1, 1])


$\textbf{NOTES}$

  > `batch_t * unsqueezed_weights` 연산원리
  > PyTorch(=NumPy) 브로드캐스팅은 뒤(마지막 차원)부터 차원을 맞춰가며 아래 조건이면 자동 확장
  >  - 각 차원에서 두 값이 같거나
  >  - 둘 중 하나가 1이면 → 1인 쪽이 “늘어나서” 맞춰짐

  > `batch_t`는 `torch.Size([2, 3, 5, 5])`, `unsqueezed_weights`는 `torch.Size([3, 1, 1])`

  > 뒤에서부터 맞춰보면 `unsqueezed_weights`의 -1번째 차원과 -2번째 차원은 5로 바뀌고(브로드캐스팅되고) -3번째 차원은 그대로, -4번째 차원이 2로 바뀐다(브로드캐스팅)
  
  > 그냥 weights는 -1번째 차원이 3이고 batch_t는 -1번째 차원이 5로 두 값이 다르고 둘 다 1이 아니므로 브로드캐스팅이 불가능

In [20]:
unsqueezed_weights = weights.unsqueeze(-1).unsqueeze(-1) # (3) -> (3, 1) -> (3, 1, 1)
img_weights = (img_t * unsqueezed_weights) # (3, 5, 5) * (3, 1, 1) -> (3, 5, 5)
batch_weights = (batch_t * unsqueezed_weights) # (2, 3, 5, 5) * (3, 1, 1) -> (2, 3, 5, 5)
img_gray_weighted = img_weights.sum(-3) # (5, 5)
batch_gray_weighted = batch_weights.sum(-3) # (2, 5, 5)
batch_weights.shape, batch_t.shape, unsqueezed_weights.shape

(torch.Size([2, 3, 5, 5]), torch.Size([2, 3, 5, 5]), torch.Size([3, 1, 1]))

In [21]:
img_gray_weighted.shape, batch_gray_weighted.shape

(torch.Size([5, 5]), torch.Size([2, 5, 5]))

In [22]:
# torch.einsum을 통한 차원 고려 덧셈
img_gray_weighted_fancy = torch.einsum('...chw,c->...hw', img_t, weights)
batch_gray_weighted_fancy = torch.einsum('...chw,c->...hw', batch_t, weights)
img_gray_weighted_fancy.shape, batch_gray_weighted_fancy.shape

(torch.Size([5, 5]), torch.Size([2, 5, 5]))

In [23]:
img_gray_weighted == img_gray_weighted_fancy

tensor([[True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True]])

위와 같이 텐서를 다룰때 차원이 커지면 텐서를 사용하는 코드 작성시에 차원을 헷갈려서 오류를 내기 쉽다. 이러한 오류를 방지하는 방법으로 차원에 이름을 부여하는 방식이 존재한다.

`tensor`나 `rand`같은 텐서 팩토리 함수에 이름으로 사용할 문자열 리스트를 `names`인자로 전달할 수 있다.

$\textbf{NOTE}$

>  만들어진지 6-7년이 넘어가는데, 아직도 pytorch 기능에 완전히 통합되지 못했다. 그냥 않써도 무방한 기능

In [24]:
weights_named = torch.tensor([0.2126, 0.7152, 0.0722], names = ['channels'])
weights_named

C:\Users\PC\AppData\Local\Temp\ipykernel_18016\2163324119.py:1: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/core/TensorImpl.h:1976.)
  weights_named = torch.tensor([0.2126, 0.7152, 0.0722], names = ['channels'])


tensor([0.2126, 0.7152, 0.0722], names=('channels',))

In [25]:
# 텐서를 먼저 만들어 나중에 이름 지정하기
img_named = img_t.refine_names(..., 'channels', 'rows', 'columns')
batch_named = batch_t.refine_names(..., 'channels', 'rows', 'columns')
print(f"img named: {img_named.shape, img_named.names}")
print(f"batch named: {batch_named.shape, batch_named.names}")

img named: (torch.Size([3, 5, 5]), ('channels', 'rows', 'columns'))
batch named: (torch.Size([2, 3, 5, 5]), (None, 'channels', 'rows', 'columns'))


텐서끼리의 연산은 먼저 각 차원의 크기가 같은지, 혹은 한쪽이 1이고 다른 쪽으로 브로드캐스팅될 수 있는지도 확인해야 한다. 이때 이름이 지정되어 있다면 파이토치가 우리를 대신해 알아서 체크해준다. 파이토치가 차원을 자동으로 정렬해주지는 않기에 우리는 명시적으로 이런 작업을 수행할 필요가 있다. `align_as` 함수는 빠진 차원을 채우고, 존재하는 차원을 올바른 순서로 바꿔준다.

$\textbf{NOTE}$

>  이것도 names와 마찬가지로, 그냥 않써도 무방한 기능

In [26]:
weights_aligned = weights_named.align_as(img_named)
weights_aligned.shape, weights_aligned.names

(torch.Size([3, 1, 1]), ('channels', 'rows', 'columns'))

In [27]:
gray_named = (img_named * weights_aligned).sum('channels')
gray_named.shape, gray_named.names

(torch.Size([5, 5]), ('rows', 'columns'))

In [28]:
# 이름 있는 텐서를 사용하는 연산을 함수 밖에서도 사용하는 방법
gray_plain = gray_named.rename(None)
gray_plain.shape, gray_plain.names

(torch.Size([5, 5]), (None, None))

## 3.5 텐서 요소 타입

  - $\textbf{파이썬에서 숫자는 객체다}$
    > 통상 부동소수점 수는 컴퓨터에서 32비트 공간을 사용한다. 하지만 파이썬은 참조 카운터까지 만들어 부동소수점 수를 완전한 파이썬 객체로 변환한다. $\textbf{박싱}^{\text{boxing}}$이라 부르는 이 연산은 수를 소량만 저장하는 경우라면 큰 문제가 없겠지만, 수백만 개가 넘어가면 상당히 비효율적이다.
  - $\textbf{파이썬에서 리스트는 연속된 객체의 컬렉션이다}$
    > 파이썬은 두 벡터의 내적을 효율적으로 수행하는 연산이 없다. 벡터 합도 마찬가지이다. 파이썬 리스트에 들어있는 데이터를 메모리에 최적화하여 배치할 방법은 딱히 없고, 리스트는 숫자 뿐만 아니라 임의의 파이썬 객체에 대한 임의 접근이 가능한 포인터의 모음이다. 게다가 파이썬 리스트는 단일 차원이며, 비록 리스트의 리스트를 만들 수도 있지만 이런 방식은 매우 비효율적이다.
  - $\textbf{파이썬 인터프리터는 최적화를 거치는 컴파일된 코드보다 느리다}$
    > 다량의 숫자 데이터 모음에 대한 수학적 연산을 수행하는 일은 C 같은 저수준 컴파일을 통해 최적화한 바이너리 코드가 훨씬 빠르다.

성능 최적화를 위해 텐서 내의 모든 객체는 같은 타입의 숫자여야 한다.

파이토치는 실행 중에 이런 숫자 타입을 계속 추적하고 있어야 한다.

### 3.5.1 dtype으로 숫자 타입 지정하기

`tensor`나 `zeros, ones`같은 텐서 생성자 실행 시 넘겨주는 `dtype`인자로 텐서 내부에 들어갈 데이터 타입(`dtype`)을 지정할 수 있다.

예 : `torch.float32`, `torch.float64`, `torch.float16`, `torch.int8`, `torch.int16`, `torch.uint16`, `torch.int32`, `torch.int64`등

텐서의 기본 데이터 타입은 32비트 부동소수점이다.

### 3.5.2 모든 경우에 사용하는 dtype

  - 신경망 연산은 대부분 32비트 부동소수점 연산이다(물론 LLM은 8비트, 4비트로 사용하기도 한다, 심하면 1비트(1.58bit)). 필요하다면 약간의 정확도를 희생해서 정밀도를 반으로 떨어뜨려 신경망이 차지하는 공간을 줄이는 방식도 가능하다.

  - 텐서는 다른 텐서에 대한 인덱스로 사용할 수 있다(인덱싱용 텐서는 64비트 정수 데이터 타입으로 간주)

  - `points > 1.0`같은 $\text{술어}^{\text{predicate}}$는 텐서 내 각각이 조건을 만족하는지 알려주는 `bool` 텐서를 만들어 낸다.(텐서 내 값은 숫자 타입)

In [29]:
# 보면 알겠지만 numpy의 where와 같다.
points = torch.randn(10, 10)
bool_cond = points > 1.0
torch_where_test = torch.where(bool_cond, 2, 0)

torch_where_test

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [2, 0, 2, 0, 0, 0, 2, 0, 0, 2],
        [0, 0, 2, 0, 0, 0, 2, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 2, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 2, 0, 2, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 2, 2, 0],
        [0, 2, 0, 0, 0, 2, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

### 3.5.3 텐서의 dtype 속성 관리

In [30]:
# 텐서에 숫자 타입 지정하는 방법1 : 생성자에 dtype인자 전달하기
double_points = torch.ones(10, 2, dtype=torch.double) # torch.float64
short_points = torch.tensor([[1,2], [3,4]], dtype=torch.short) # torch.int16

print(short_points.dtype)

torch.int16


In [31]:
# 텐서에 숫자 타입 지정하는 방법2 : 캐스팅 메소드 사용
double_points = torch.zeros(10, 2).double()
short_points = torch.tensor([[1,2], [3,4]]).short() # torch.int16

In [32]:
# 텐서에 숫자 타입 지정하는 방법3 : to 메소드 사용 (변환이 필요한 경우에만 진행, 그렇지 않으면 생성자 단계부터 바로 dtype 지정하는 방법1이 더 빠름)
double_points = torch.zeros(10, 2).to(torch.double)
short_points = torch.tensor([[1,2], [3,4]]).to(dtype=torch.short)

In [33]:
# 여러 타입을 가진 입력들이 연산을 거치면서 섞일 때 자동으로 제일 큰 타입으로 연산 진행
points_64 = torch.rand(5, dtype = torch.double)
points_short = points_64.to(torch.short)
points_64 * points_short

tensor([0., 0., 0., 0., 0.], dtype=torch.float64)

## 3.6 텐서 API

파이토치는 다양한 텐서 연산을 제공하며 자세한건 https://pytorch.org/docs 온라인문서 참조.

텐서끼리의 연산 대부분은 `torch`모듈에 있고 대부분이 텐서 객체에 대해 메소드처럼 호출할 수 있다.

In [34]:
a = torch.ones(3,2)
a_t = torch.transpose(a, 0, 1)

a.shape, a_t.shape

(torch.Size([3, 2]), torch.Size([2, 3]))

In [35]:
# 텐서 메소드
a = torch.ones(3,2)
a_t = a.transpose(0, 1)

a.shape, a_t.shape

(torch.Size([3, 2]), torch.Size([2, 3]))

## 3.7 텐서를 저장소 관점에서 머릿속에 그려보기

  - 텐서 내부 값은 실제로는 `torch.Storage`인스턴스로 관리되며 연속적인 메모리 조각으로(memory allocation)할당된 상태다.
  - 저장 공간은 숫자 데이터를 가진 1차원 배열이다.
  - 부동소수점 수를 나타내기 위한 32비트 공간의 `float`나 정수를 표현하기 위한 64비트 공간의 `int64`타입으로 된 숫자들이 연속으로 들어 있는 메모리 블럭으로 보면 된다.

  - 파이토치의 `Tensor`객체는 이러한 저장 공간을 나타내는 `Storage`객체에 대한 뷰 역할을 담당하고
  - 오프셋을 사용해서 공간의 임의 위치에 접근하거나 특정 차원의 크기를 단위로 해서 접근할 수 있다.

  - `reshape`등으로 동일한 저장공간을 참조하는 여러 개의 텐서를 만들어도, 이건 새로운 `Storage`를 만드는게 아닌 기존 `Storage`에 대한 뷰를 만드는거라 데이터 크기에 상관없이 빠르게 수행됨(make sense)


$\textbf{NOTE}$

  > 현재는 UntypedStorage로 Storage가 실제 컴퓨터 상에서 무슨 값을 담고 있는지 직접 확인가능하며, 이 값을 직접 바꾸는건 권장되지 않음(low-level manipulation이므로 전혀 예상하지 못한 결과가 나올 수 있음)

### 3.7.1 저장 공간 인덱싱

텐서를 위한 저장 공간은 `.storage` 메소드로 접근할 수 있다.

In [36]:
points = torch.tensor([[4.0, 1.0], [5.0, 3.0], [2.0, 1.0]])
points.storage()

C:\Users\PC\AppData\Local\Temp\ipykernel_18016\2923139482.py:2: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  points.storage()


 4.0
 1.0
 5.0
 3.0
 2.0
 1.0
[torch.storage.TypedStorage(dtype=torch.float32, device=cpu) of size 6]

In [37]:
points.untyped_storage()

 0
 0
 128
 64
 0
 0
 128
 63
 0
 0
 160
 64
 0
 0
 64
 64
 0
 0
 0
 64
 0
 0
 128
 63
[torch.storage.UntypedStorage(device=cpu) of size 24]

In [38]:
points_storage = points.storage()
points_storage[0]

4.0

In [39]:
points.storage()[1]

1.0

In [40]:
# 저장 공간에 직접 접근해서 값 바꾸기
points = torch.tensor([[4.0, 1.0], [5.0, 3.0], [2.0, 1.0]])
points_storage = points.storage()
points_storage[0] = 2.0
points

tensor([[2., 1.],
        [5., 3.],
        [2., 1.]])

### 3.7.2 저장된 값을 수정하기 : 텐서 내부 연산

in-place 연산 방식

In [41]:
# zero_ 메소드
a = torch.ones(3,2)
a.zero_()
a

tensor([[0., 0.],
        [0., 0.],
        [0., 0.]])

## 3.8 텐서 메타데이터 : 사이즈, 오프셋, 스트라이드

저장 공간을 인덱스로 접근하기 위해 텐서는 저장 공간에 포함된 몇 가지 명확한 정보, 즉 $\text{사이즈}^{\text{size}}$, $\text{오프셋}^{\text{offset}}$, $\text{스트라이드}^{\text{stride}}$에 의존한다.

  - 사이즈 : 텐서의 각 차원 별로 들어가는 요소의 수를 표시한 튜플
  - 오프셋 : 텐서의 첫 번째 요소를 가리키는 색인 값과 동일
  - 스트라이드 : 각 차원에서 다음 요소를 가리키고 싶을 때 실제 저장 공간상에서 몇 개의 요소를 건너뛰어야 하는지 알려주는 숫자

In [42]:
torch_meta_test = torch.randn(3, 5, 5) 
# offset = 0, size = (3, 5, 5), stride = (25, 5, 1)
# offset은 그냥 텐서를 1차원 배열의 storage로 보았을 때 해당하는 요소의 위치 값과 동일
# stride는 다음 요소를 찾을 때 몇칸씩 건너뛰어야하는지를 차원별로 알려주는 숫자

In [43]:
torch_meta_test.storage_offset()

0

In [44]:
torch_meta_test.stride()

(25, 5, 1)

### 3.8.1 다른 텐서의 저장 공간에 대한 뷰 만들기

In [45]:
points = torch.tensor([4.0, 1.0, 5.0, 3.0, 2.0, 1.0]).reshape(3,2)
second_point = points[1]
second_point.storage_offset()

2

In [46]:
second_point 

tensor([5., 3.])

In [47]:
points.storage_offset()

0

In [48]:
second_point.size() # 텐서 객체의 shape 속성값과 동일하다

torch.Size([2])

In [49]:
second_point.shape

torch.Size([2])

In [50]:
# 2차원 텐서에서 요소 i,j에 접근한다면 저장 공간상으로는
# storage_offset + stride[0] * i + stride[1] * j 번째 요소가 됨

`Tensor`를 통해 `Storage`를 간접 접근하는 방식으로 텐서의 전치(Transpose)나 일부만으로 더 작은 텐서를 만드는데 필요한 연산을 저렵하게 수행할 수 있다(메모리 재할당을 하지 않으므로). 메모리 재할당 대신, 사이즈, 저장소 오프셋, 스트라이드를 변경한 새로운 `Tensor`객체를 할당하는 과정을 포함한다.

In [51]:
# 중요한 점은 새로운 텐서가 원래의 points 텐서가 가리키는 저장 공간과 동일한 저장 공간을 기리킨다는 점
second_point = points[1]
print(second_point.size())
print(second_point.storage_offset())
print(second_point.stride())

torch.Size([2])
2
(1,)


In [52]:
# 그래서 새로운 텐서를 변경하면 기존 텐서 값도 바뀐다.
points = torch.tensor([4.0, 1.0, 5.0, 3.0, 2.0, 1.0]).reshape(3,2)
second_point = points[1]
second_point[0] = 10.0
points

tensor([[ 4.,  1.],
        [10.,  3.],
        [ 2.,  1.]])

In [53]:
# 위와 같은 뷰 기반이라 그렇다. 기존 텐서를 그대로 냅두려면 copy를 하면 되는데 파이토치는 clone이라는 메소드를 쓴다
points = torch.tensor([4.0, 1.0, 5.0, 3.0, 2.0, 1.0]).reshape(3,2)
second_point = points[1].clone()
second_point[0] = 10.0
points

tensor([[4., 1.],
        [5., 3.],
        [2., 1.]])

### 3.8.2 복사 없이 텐서 전치하기

`points`텐서에는 행마다 개별 포인트가 들어 있고 열에는 $X$, $Y$좌표가 들어 있다. 이를 전치하여 열에 포인트가 들어가도록 만들 수 있다.

메소드는 `transpose`혹은 `t`를 사용한다.

In [54]:
points = torch.tensor([4.0, 1.0, 5.0, 3.0, 2.0, 1.0]).reshape(3,2)
points

tensor([[4., 1.],
        [5., 3.],
        [2., 1.]])

In [55]:
points_t = points.t()
points_t

tensor([[4., 5., 2.],
        [1., 3., 1.]])

In [56]:
# 두 텐서가 같은 공간을 가리키는지 확인
id(points.storage()) == id(points_t.storage())

False

In [57]:
# 두 텐서는 차원 정보와 스트라이드만 다르다.
print(points.stride())
print(points_t.stride())
print(points.size())
print(points_t.size())

(2, 1)
(1, 2)
torch.Size([3, 2])
torch.Size([2, 3])


### 3.8.3 더 높은 차원에서의 전치 연산

  - 파이토치의 전치연산은 2차원 배열형태인 행렬뿐 아니라 다차원 배열에 대해서도 가능하다.
  - 차원 정보와 스트라이드가 바뀔 두차원을 각각 지정해주면 전치 된다.

  - 가장 오른쪽 차원에서 시작해서 증가되는 형태로 저장소에 값이 펼쳐진 텐서(2차원 텐서에서 열을 따라 이동)는 `contiguous`로 정의된다.
  - 인접한(contiguous) 텐서는 값 순회 시 띄엄띄엄 참조하지 않기 때문에 데이터 지역성(data locality)관점에서 CPU 메모리 접근 효율이 좋다.
  - 실제 효율 여부는 이를 사용하는 알고리즘에 따라 다를 수 있다.

In [58]:
some_t = torch.ones(3, 4, 5)
transpose_t = some_t.transpose(0, 2)
print(some_t.shape) # (3, 4, 5)
print(transpose_t.shape) # (5, 4, 3)
print(some_t.stride()) # (20, 5, 1)
print(transpose_t.stride()) # (12, 3, 1)

torch.Size([3, 4, 5])
torch.Size([5, 4, 3])
(20, 5, 1)
(1, 5, 20)


### 3.8.4 인접한 텐서

  - 파이토치 텐서 연산 중에는 인접한 텐서에 대해서만 동작하는 경우가 있다(예 : view).
  - 이러한 경우 파이토치는 경고성 예외를 던져서 명시적으로 `contiguous`를 호출하라고 알려준다.
  - 텐서가 이미 인접한 상태라면 `contiguous`가 실제로 하는 일은 없고 성능에 지장을 주지도 않는다.

In [59]:
# 인접하지 않은 텐서의 예시
print(points.is_contiguous())
print(points_t.is_contiguous())

True
False


In [60]:
# contiguous를 통한 인접하지 않은 텐서 인접하게 만들기 : 텐서 내용은 동일하나, 값의 배치와 스트라이드가 바뀐 텐서 생성됨
points = torch.tensor([[4.0, 1.0], [5.0, 3.0], [2.0, 1.0]])
points_t = points.t()
print(points_t)
print(points_t.storage())
print(points_t.stride())

print("\n")
points_t_cont = points_t.contiguous()
print(points_t_cont)
print(points_t_cont.storage())
print(points_t_cont.stride())

tensor([[4., 5., 2.],
        [1., 3., 1.]])
 4.0
 1.0
 5.0
 3.0
 2.0
 1.0
[torch.storage.TypedStorage(dtype=torch.float32, device=cpu) of size 6]
(1, 2)


tensor([[4., 5., 2.],
        [1., 3., 1.]])
 4.0
 5.0
 2.0
 1.0
 3.0
 1.0
[torch.storage.TypedStorage(dtype=torch.float32, device=cpu) of size 6]
(3, 1)


## 3.9 텐서를 GPU로 옮기기

  - 파이토치의 텐서는 CPU 외 다른 프로세서인 GPU를 위한 메모리 공간에도 저장할 수 있다.
  - 모든 파이토치 텐서는 여러 GPU 중에 특정 GPU 메모리 영역으로 이동할 수 있고 이렇게 대량 병렬 처리 연산을 빠르게 수행할 수 있다.
  - 이때의 텐서 연산은 파이토치가 지원하는 GPU 전용 루틴을 사용한다.

### 3.9.1 텐서 디바이스 속성 관리

`dtype`외에 파이토치의 `Tensor`는 `device`라는 인자로 텐서 데이터가 실제 컴퓨터의 어디에 위치할지도 지정할 수 있다. 다음은 생성자에 `device`인자를 지정하여 텐서를 GPU에 만드는 방법이다.

In [61]:
points_gpu = torch.tensor([4.0, 1.0, 5.0, 3.0, 2.0, 1.0], device = 'cuda').reshape(3,2)
points_gpu

tensor([[4., 1.],
        [5., 3.],
        [2., 1.]], device='cuda:0')

In [62]:
# to 메소드를 통해 CPU에 만들어진 텐서를 GPU로 옮기기
points_gpu = points.to(device='cuda')
points_gpu

tensor([[4., 1.],
        [5., 3.],
        [2., 1.]], device='cuda:0')

In [63]:
# 컴퓨터에 GPU가 두 개 이상 탑재된 경우 어떤 GPU에서 동작시킬지 정수번호로 지정 가능
points_gpu = points.to(device='cuda:0')
points_gpu

tensor([[4., 1.],
        [5., 3.],
        [2., 1.]], device='cuda:0')

In [64]:
# 계산 예제
points = 2 * points # CPU에서 실행되는 곱셈
points_gpu = 2 * points.to(device='cuda') # GPU에서 실행되는 곱셈

In [65]:
# GPU에 들어있는 텐서 CPU로 옮기기
points_cpu = points_gpu.to(device='cuda')

In [66]:
# to 메소드 대신 cpu나 cuda메소드도 가능
points_gpu = points.cuda()
points_gpu = points.cuda(0)
points_cpu = points_gpu.cpu()

In [67]:
# to 메소드 사용시 device와 dtype 동시에 사용 가능
points_gpu_f16 = points.to(device='cuda', dtype=torch.float16)
points_gpu_f16

tensor([[ 8.,  2.],
        [10.,  6.],
        [ 4.,  2.]], device='cuda:0', dtype=torch.float16)

## 3.10 넘파이 호환

  - 파이토치와 넘파이 배열의 변환은 매우 효율적으로 이뤄진다.
  - 넘파이 배열과의 제로카피(zero-copy)수준의 상호 변환은 파이썬 버퍼 프로토콜이라는 저장 시스템 덕분이다.

In [68]:
# points 텐서를 넘파이 배열로 바꾸는 방법
points = torch.ones(3,4)
points_np = points.numpy()
points_np

array([[1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.]], dtype=float32)

  - 위의 `points_np` 배열은 기존 텐서의 저장 공간을 그대로 버퍼로 사용한다. 즉 데이터가 CPU RAM 영역에 있는 한, `numpy`메소드를 아무 추가 비용 없이 실행할 수 있다.
  - 넘파이 배열을 수정하면 원래 텐서에도 그대로 적용된다.
  - 만일 텐서가 GPU에 할당되어 있다면 파이토치는 넘파이 배열로의 변환 과정에서 CPU 영역으로 텐서 내용을 복사해서 배열을 만든다(이러한 과정이 최소화 되어야 연산속도가 빨라지는 것).

In [69]:
# 넘파이 배열을 파이토치 텐서로 만들기
points = torch.from_numpy(points_np)
points

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

In [70]:
# numpy vs torch(gpu) 연산 비교
rand_mat1 = torch.randn(64, 100, 256, 256)
rand_mat2 = torch.randn(1, 1, 256, 256)

rand_mat1_gpu = rand_mat1.to(device='cuda', dtype = torch.float32)
rand_mat1_cpu = rand_mat1.numpy()

rand_mat2_gpu = rand_mat2.to(device='cuda', dtype = torch.float32)
rand_mat2_cpu = rand_mat2.numpy()

In [71]:
%time rand_mat1_cpu @ rand_mat2_cpu

CPU times: total: 17.6 s
Wall time: 2.48 s


array([[[[-2.06017723e+01,  9.10524750e+00,  1.56067734e+01, ...,
           5.42208767e+00,  7.42066002e+00, -2.17035046e+01],
         [-1.67214813e+01, -5.69319057e+00,  9.55311489e+00, ...,
          -1.15875883e+01,  3.97224350e+01, -2.31438427e+01],
         [-9.87429047e+00,  1.01532125e+01, -2.99801941e+01, ...,
           9.16710281e+00,  1.68058414e+01, -4.16269493e+01],
         ...,
         [-2.24887013e+00, -1.71315217e+00,  9.49340439e+00, ...,
           1.11388597e+01, -9.07029724e+00,  6.00289917e+00],
         [-2.28216438e+01, -2.04706898e+01, -4.29383427e-01, ...,
           5.65720463e+00,  9.43895435e+00,  1.30382423e+01],
         [-1.66598129e+01, -9.31069565e+00,  2.52361450e+01, ...,
          -2.50959320e+01,  9.82110882e+00, -5.18270731e+00]],

        [[ 1.46637859e+01, -1.71698618e+00,  7.07226276e+00, ...,
          -4.93491821e+01, -6.26537037e+00, -4.25716705e+01],
         [ 5.86955929e+00, -1.03287401e+01,  1.74802136e+00, ...,
          -1.20490370e

In [72]:
%time rand_mat1_gpu @ rand_mat2_gpu

CPU times: total: 109 ms
Wall time: 98 ms


tensor([[[[-2.0602e+01,  9.1052e+00,  1.5607e+01,  ...,  5.4221e+00,
            7.4207e+00, -2.1704e+01],
          [-1.6721e+01, -5.6932e+00,  9.5531e+00,  ..., -1.1588e+01,
            3.9722e+01, -2.3144e+01],
          [-9.8743e+00,  1.0153e+01, -2.9980e+01,  ...,  9.1671e+00,
            1.6806e+01, -4.1627e+01],
          ...,
          [-2.2489e+00, -1.7132e+00,  9.4934e+00,  ...,  1.1139e+01,
           -9.0703e+00,  6.0029e+00],
          [-2.2822e+01, -2.0471e+01, -4.2938e-01,  ...,  5.6572e+00,
            9.4390e+00,  1.3038e+01],
          [-1.6660e+01, -9.3107e+00,  2.5236e+01,  ..., -2.5096e+01,
            9.8211e+00, -5.1827e+00]],

         [[ 1.4664e+01, -1.7170e+00,  7.0723e+00,  ..., -4.9349e+01,
           -6.2654e+00, -4.2572e+01],
          [ 5.8696e+00, -1.0329e+01,  1.7480e+00,  ..., -1.2049e+01,
            2.0747e-01, -1.9517e+01],
          [ 7.9628e+00, -5.5010e-01, -1.2868e+01,  ..., -1.2223e+01,
            1.8519e-01, -6.1307e-01],
          ...,
     

## 3.11 일반화된 텐서도 텐서다

  - 파이토치는 우리의 텐서가 CPU에 있꺼나 GPU에 있거나 상관없이 알맞은 연산 함수를 호출한다($\textbf{분배}^{\text{dispatching}}$ 메커니즘).
  - 분배 메커니즘은 사용자 API 입장에서 해당 텐서 타입에 맞는 백엔드 함수를 골라서 실행한다(구글의 TPU같은 장치에 특화된 텐서는 TPU용 백엔드 함수 사용)
  - $\text{희소}^{\text{sparse}}$ 텐서는 0이 아닌 경우만 인덱스와 값을 만들어서 저장
  - 파이토치 분배기는 확장 가능하도록 설계되었으며, 내부의 스위치 로직으로 백엔드 별로 구현된 각각의 코드 중에 선택하여 다양한 숫자 타입을 다룰 수 있게 되어 있음
  - 양자화(quantized)된 텐서도 존재(백엔드 연산을 특별하게 수행하도록 구현된 텐서).

## 3.12 텐서 직렬화

텐서를 로컬영역에 저장하는 방법을 다룬다.

In [73]:
import os, sys

BASE_DIR = os.getcwd()
DATA_PATH = os.path.join(BASE_DIR, "data")
save_path = os.path.join(DATA_PATH, "p1ch3", "ourpoints.t")
torch.save(points, save_path)

In [74]:
# 혹은 파일 이름 대신 파일 디스크립터(descriptor)넣기
with open(save_path, 'wb') as f:
    torch.save(points, f)

In [75]:
# 텐서 불러오기
points = torch.load(save_path)

# 혹은 파일 디스크립터
with open(save_path, 'rb') as f:
    points = torch.load(f)

### 3.12.1 h5py로 HDF5 병렬화하기

`torch`파일은 파이토치가 아닌 다른 소프트웨어로는 읽을 수 없으므로, 텐서를 호환 가능하도록 저장하는 방법을 알아야 한다.

호환성이 중요한 경우 HDF5 포맷을 위한 라이브러리가 필요하다(www.hdfgroup.org/solutions/hdf5). 

HDF5는 이식성이 높고, 광범위하게 지원되는, 중첩된 key-value 딕셔너리에서 직렬화된 정형 다차원 배열을 표현하는 포맷이다.

파이썬은 `h5py` 라이브러리를 통해 HDF5 포맷을 지원한다(www.h5py.org).

이 라이브러리는 넘파이 배열 형태로 전달하거나 반환받을 수 있다.

In [76]:
# points 텐서 넘파이 배열로 변환해서 저장하기
import h5py

out_path = os.path.join(DATA_PATH, "p1ch3", "ourpoints.hdf5")
f = h5py.File(out_path, 'w')
dset = f.create_dataset('coords', data = points.numpy())
f.close()

위에서 `coords`가 HDF5 파일의 키에 해당하는데 키와 키를 $\text{중첩}^{\text{nested}}$시킬 수도 있다. 

HDF5 포맷의 재미있는 특징 중 하나는 데이터가 디스크에 있는 상태에서 원하는 요소만 인덱스로 접근하도록 지원해주는 점이다.

In [77]:
# 데이터셋의 가장 뒤 포인트 두 개만 읽는 경우
f = h5py.File(out_path, 'r')
dset = f['coords']
last_points = dset[-2:]

# 텐서 저장공간으로 복사하기
last_points = torch.from_numpy(dset[-2:])
# HDF5 파일 닫기 : 닫은 이후로는 데이터셋이 무효화되어 이후부터는 dset 접근시 예외 발생
f.close()

print(last_points, last_points.shape, type(last_points))

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.]]) torch.Size([2, 4]) <class 'torch.Tensor'>


## 3.13 결론

  - 부동소수점 수 표현을 위해 필요한 모든 것을 살펴보았다.
  - 텐서의 뷰, 다른 텐서로 텐서 인덱싱하기, 다른 사이즈나 차원 정보를 가지는 텐서 사이에서 요소 단위 연산 단순화시켜주는 브로드캐스팅하기 등

## 3.14 연습문제

1. `list(range(9))`으로부터 텐서를 만들어보자. 사이즈, 오프셋, 스트라이드는 얼마일지 계산해보자.

    - a. `a.view(3,3)`으로 텐서를 만들어라. `view`의 역할은 무엇인가? `a`와 `b`가 같은 공간을 가리키고 있는지 확인해보자.
    - b. `c = b[1:, 1:]`으로 텐서를 만들고, 사이즈, 오프셋, 스트라이드는 얼마일지 계산해보자.

2. 코사인이나 제곱근같은 수학 연산을 하나 골라라. 동일한 역할을 하는 함수를 `torch`라이브러리에서 찾아보자.

    - a. 텐서 `a`에 대해 해당 함수를 요소 단위로 실행해보자. 왜 오류가 발생할까?
    - b. 동작시키려면 어떤 연산이 필요할까?
    - c. 해당 연산을 추가 공간을 사용하지 않고 실행하는 함수가 있을까?

In [ ]:
# 1.
python_data = list(range(9))
a = torch.tensor(python_data)
print(a.size()) # (9,)
print(a.storage_offset()) # 0
print(a.stride()) # 1

b = a.view(3, 3) # view의 역할은 새로운 storage를 만들지 않고, 기존 텐서를 참조하여 불필요한 메모리 재할당을 하지 않게끔 한다.
print(b.size()) # (3, 3)
print(b.storage_offset()) # 0
print(b.stride()) # (3, 1)
print(id(a.storage()) == id(b.storage())) # True -> 틀렸다. 이유는 

c = b[1:, 1:]
print(c.size()) # (2, 2)
print(c.storage_offset()) # 4
print(c.stride()) # (3, 1) 혹은 (2, 1) (근데 새로운 storage 만드는게 아니니까 3,1 일것 같다)

torch.Size([9])
0
(1,)
torch.Size([3, 3])
0
(3, 1)
False
torch.Size([2, 2])
4
(3, 1)


In [132]:
# 2.
cosine_list = []
for i in range(len(a)):
    x = torch.cos(a[i])
    cosine_list.append(x)

In [146]:
import numpy as np
import math

long_num_list = np.linspace(0, 1, 1_000_000)
long_num_torch = torch.tensor(np.linspace(0, 1, 1_000_000))
long_num_numpy = np.linspace(0, 1, 1_000_000)

def for_loop_test(long_num_list):
    cosine_list = []
    for i in range(len(long_num_list)):
        x = math.cos(long_num_list[i])
        cosine_list.append(x)    
    # return cosine_list

def torch_test(long_num_torch):
    x = torch.cos(long_num_torch)
    # return x   

def numpy_test(long_num_numpy):
    x = np.cos(long_num_numpy)
    # return x

In [147]:
%time for_loop_test(long_num_list)

CPU times: total: 125 ms
Wall time: 138 ms


In [148]:
%time torch_test(long_num_torch)

CPU times: total: 0 ns
Wall time: 1.42 ms


In [149]:
%time numpy_test(long_num_numpy)

CPU times: total: 31.2 ms
Wall time: 3.5 ms


In [151]:
import torch, os
print("torch num threads:", torch.get_num_threads())
print("OMP_NUM_THREADS:", os.environ.get("OMP_NUM_THREADS"))
print("MKL_NUM_THREADS:", os.environ.get("MKL_NUM_THREADS"))
print("numpy:", np.show_config())

torch num threads: 14
OMP_NUM_THREADS: None
MKL_NUM_THREADS: None
Build Dependencies:
  blas:
    detection method: pkgconfig
    found: true
    include directory: C:/Users/PC/anaconda3/envs/mini-synth/Library/include
    lib directory: C:/Users/PC/anaconda3/envs/mini-synth/Library/lib
    name: mkl-sdl
    openblas configuration: unknown
    pc file directory: C:\miniconda3\conda-bld\numpy_and_numpy_base_1763980696204\_h_env\Library\lib\pkgconfig
    version: '2025'
  lapack:
    detection method: pkgconfig
    found: true
    include directory: C:/Users/PC/anaconda3/envs/mini-synth/Library/include
    lib directory: C:/Users/PC/anaconda3/envs/mini-synth/Library/lib
    name: mkl-sdl
    openblas configuration: unknown
    pc file directory: C:\miniconda3\conda-bld\numpy_and_numpy_base_1763980696204\_h_env\Library\lib\pkgconfig
    version: '2025'
Compilers:
  c:
    commands: cl.exe
    linker: link
    name: msvc
    version: 19.29.30159
  c++:
    commands: cl.exe
    linker: link

## 3.15 핵심 요약

- 신경망은 부동소수점 표현을 다른 부동소수점 표현으로 바꿔준다. 입력이나 출력 부분의 값들은 사람이 확인 가능하지만 중간 단계의 값은 파악하기 어렵다.
- 부동소수점 표현은 텐서에 저장한다.
- 텐서는 다차원 배열이며 파이토치의 기본 자료구조이다.
- 파이토치는 텐서의 생성이나 조작 그리고 수학 연산 같은 다양한 표준 라이브러리를 제공한다.
- 텐서는 병렬화해 디스크에 저장하거나 반대로 불러올 수 있다.
- 파이토치의 모든 텐서 연산은 코드 변경 없이 쉽게 CPU에서 혹은 GPU에서 실행할 수 있다.
- 파이토치에서 바꿔치기 연산을 수행하는 함수 이름은 `Tensor.sqrt_`처럼 밑줄(_)로 끝난다.